[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/01_tensor_memory_layout_and_ops.ipynb)

# 01. Tensor memory layout and core ops

뒤의 모든 노트북에서 반복해서 등장하는 tensor shape, stride, view, transpose, broadcast, reduction, gather/scatter, matmul의 실제 동작을 작은 숫자로 확인한다.

**반복 형식:** 바닐라 PyTorch 실행 → profiler로 ATen/CUDA 연산 확인 → 필요할 때만 작은 텐서로 수학적 전개를 펼친다.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)


In [ ]:
from torch.profiler import profile, ProfilerActivity

def profile_call(name, fn, *args, **kwargs):
    activities = [ProfilerActivity.CPU]
    if torch.cuda.is_available():
        activities.append(ProfilerActivity.CUDA)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    with profile(
        activities=activities,
        record_shapes=True,
        profile_memory=True,
        with_stack=False,
    ) as prof:
        out = fn(*args, **kwargs)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    print(f"\n[{name}] top operators")
    sort_key = "self_cuda_time_total" if torch.cuda.is_available() else "self_cpu_time_total"
    print(prof.key_averages().table(sort_by=sort_key, row_limit=12))

    return out


## 1. Shape, stride, storage


In [ ]:
x = torch.arange(12, device=device).reshape(3, 4)
print(x)
print("shape:", x.shape)
print("stride:", x.stride())
print("contiguous:", x.is_contiguous())


## 2. View / reshape / transpose / contiguous


In [ ]:
xt = x.transpose(0, 1)
print("transposed stride:", xt.stride())
print("transposed contiguous:", xt.is_contiguous())

xc = xt.contiguous()
print("after contiguous stride:", xc.stride())
print("after contiguous:", xc.is_contiguous())

flat = xc.view(-1)
print("flat:", flat)


In [ ]:
_ = profile_call("transpose only", lambda z: z.transpose(0, 1), x)
_ = profile_call("transpose + contiguous", lambda z: z.transpose(0, 1).contiguous(), x)


## 3. Broadcasting and reduction


In [ ]:
a = torch.tensor([[1.0], [2.0], [3.0]], device=device)
b = torch.tensor([[10.0, 20.0, 30.0, 40.0]], device=device)

y = a + b
print(y)
print("row mean:", y.mean(dim=1))
print("column sum:", y.sum(dim=0))


In [ ]:
_ = profile_call("broadcast add + reduction", lambda p, q: (p + q).mean(dim=1), a, b)


## 4. Indexing, gather, scatter


In [ ]:
src = torch.tensor([[10., 11., 12.], [20., 21., 22.]], device=device)
idx = torch.tensor([[2, 0], [1, 2]], device=device)

g = torch.gather(src, dim=1, index=idx)
print("gather result:\n", g)

base = torch.zeros(2, 3, device=device)
s = base.scatter(1, idx, g)
print("scatter result:\n", s)


In [ ]:
_ = profile_call("gather", lambda z, i: torch.gather(z, 1, i), src, idx)
_ = profile_call("scatter", lambda z, i, v: z.scatter(1, i, v), base, idx, g)


## 5. matmul / bmm / einsum


In [ ]:
A = torch.arange(12, dtype=torch.float32, device=device).reshape(3, 4)
B = torch.arange(8, dtype=torch.float32, device=device).reshape(4, 2)

print("matmul:\n", A @ B)

Ab = A.unsqueeze(0).repeat(2, 1, 1)
Bb = B.unsqueeze(0).repeat(2, 1, 1)
print("bmm shape:", torch.bmm(Ab, Bb).shape)

print("einsum equals matmul:",
      torch.allclose(torch.einsum("ik,kj->ij", A, B), A @ B))


In [ ]:
_ = profile_call("matmul", lambda p, q: p @ q, A, B)
_ = profile_call("bmm", lambda p, q: torch.bmm(p, q), Ab, Bb)
_ = profile_call("einsum", lambda p, q: torch.einsum("ik,kj->ij", p, q), A, B)


## References and provenance

**[1.1] Tensor views and strides**
- 출처: PyTorch tensor semantics / ATen
- 이 노트북에서 가져온 부분: shape·stride와 view의 관계

**[1.2] Gather / scatter**
- 출처: PyTorch ATen indexing operators
- 이 노트북에서 가져온 부분: MoE와 3D scatter에서 반복되는 핵심 연산
